# Exportar o modelo treinado para GGUF (rodar no Colab)

O adaptador LoRA sozinho não roda em `llama.cpp`/Ollama: é preciso **fundi-lo ao
modelo base** e depois **quantizar**. A fusão carrega o Qwen3-8B em 16 bits, o que
pede ~17 GB de memória — por isso o passo acontece aqui, e não na máquina local.

O resultado é um único arquivo `.gguf` de ~5 GB que vai para o Drive.

**Runtime necessário:** GPU com 20 GB+ (A100, L4) **ou** CPU com 25 GB+ de RAM
(Colab Pro, opção "High-RAM"). Uma T4 de 16 GB não dá conta da fusão na GPU, mas
a célula cai para a CPU automaticamente se houver RAM.

In [ ]:
# 1. Drive e caminhos
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

MODELO_BASE = "Qwen/Qwen3-8B"
PROJETO = Path("/content/drive/MyDrive/VFP-LLM-Qwen3-8B-2")   # ajuste se o seu for outro
ADAPTADOR = PROJETO / "final_model"
SAIDA_DRIVE = PROJETO / "gguf"

FUNDIDO = Path("/content/fundido")        # modelo fundido em 16 bits (temporário, ~16 GB)
GGUF_DIR = Path("/content/gguf")          # gguf f16 + quantizados (temporário)

# Quais quantizações gerar. Q4_K_M é o padrão; Q3_K_M é o plano B para
# máquinas com 8 GB de RAM, onde o Q4 vive no limite e força swap.
QUANTIZACOES = ["Q4_K_M", "Q3_K_M"]

for pasta in (FUNDIDO, GGUF_DIR):
    pasta.mkdir(parents=True, exist_ok=True)
SAIDA_DRIVE.mkdir(parents=True, exist_ok=True)

assert ADAPTADOR.exists(), f"adaptador não encontrado em {ADAPTADOR}"
print("adaptador:", ADAPTADOR)
print("arquivos:", sorted(p.name for p in ADAPTADOR.iterdir()))

In [ ]:
# 2. Onde a fusão cabe: GPU, CPU ou nenhuma das duas
import subprocess, torch, psutil

ram_gb = psutil.virtual_memory().total / 1e9
vram_gb = 0.0
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB")
else:
    print("GPU: nenhuma")
print(f"RAM: {ram_gb:.1f} GB")
print(f"disco livre: {psutil.disk_usage('/content').free / 1e9:.1f} GB")

if vram_gb >= 20:
    DISPOSITIVO = "cuda"
elif ram_gb >= 25:
    DISPOSITIVO = "cpu"
else:
    raise SystemExit(
        "sem memória para a fusão. Troque o runtime para A100/L4, ou para "
        "CPU com High-RAM (Ambiente de execução > Alterar tipo de ambiente)."
    )
print("fusão será feita em:", DISPOSITIVO)

In [ ]:
# 3. Dependências
!pip -q install -U "transformers>=4.51" peft accelerate safetensors sentencepiece

In [ ]:
# 4. Fundir o adaptador LoRA ao modelo base
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={"": 0} if DISPOSITIVO == "cuda" else None,
)

modelo = PeftModel.from_pretrained(base, str(ADAPTADOR))
modelo = modelo.merge_and_unload()
modelo.save_pretrained(str(FUNDIDO), safe_serialization=True, max_shard_size="4GB")

# O tokenizador vem da pasta do adaptador para preservar o chat template do treino.
tok = AutoTokenizer.from_pretrained(str(ADAPTADOR))
tok.save_pretrained(str(FUNDIDO))

del modelo, base
torch.cuda.empty_cache()

for arquivo in sorted(FUNDIDO.iterdir()):
    print(f"{arquivo.name:45} {arquivo.stat().st_size / 1e6:8.1f} MB")

In [ ]:
# 5. llama.cpp: só o conversor e o quantizador
import os
if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF > /tmp/cmake.log 2>&1
!cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-quantize >> /tmp/cmake.log 2>&1
!ls -la /content/llama.cpp/build/bin/llama-quantize || tail -30 /tmp/cmake.log

In [ ]:
# 6. Converter para GGUF em 16 bits (arquivo grande e intermediário)
GGUF_F16 = GGUF_DIR / "vfp9-qwen3-8b-f16.gguf"
!python /content/llama.cpp/convert_hf_to_gguf.py {FUNDIDO} --outfile {GGUF_F16} --outtype f16
print(f"{GGUF_F16.name}: {GGUF_F16.stat().st_size / 1e9:.2f} GB")

In [ ]:
# 7. Quantizar e copiar para o Drive
import shutil

QUANTIZADOR = "/content/llama.cpp/build/bin/llama-quantize"
for tipo in QUANTIZACOES:
    destino = GGUF_DIR / f"vfp9-qwen3-8b-{tipo.lower()}.gguf"
    !{QUANTIZADOR} {GGUF_F16} {destino} {tipo}
    final = SAIDA_DRIVE / destino.name
    shutil.copy2(destino, final)
    print(f"-> {final} ({final.stat().st_size / 1e9:.2f} GB)")

print("\npronto. Baixe do Drive:", SAIDA_DRIVE)

## Próximo passo (na máquina local)

Baixe o `.gguf` do Drive e registre no Ollama:

```bash
ollama create vfp9 -f Modelfile
ollama run vfp9
```

O `Modelfile` está em `ollama/Modelfile` no repositório.